# OMNI-X23 · Held-out UCF-Crime measured like training: event windows shifted 8 s earlier (Colab GPU)

D-66 (committed before this notebook first ran). Train and validation windows end 8 s before the event (D-49);
held-out's ended at it. Here held-out UCF-Crime's event windows are rebuilt as training's: **30 s
`[event − 38, event − 8)`, 60 s `[event − 68, event − 8)`** (`datasets/windows/ucf_crime_heldout_shift8.csv`,
`datasets/samples/ucf_crime_heldout_shift8.csv`, `scripts/build_heldout_shift.py`). Normal windows and CCD unchanged.

- **The served model stays `pfm-v5a-s1-e14`** (D-65), with validation's `θ_d`. Nothing is refit or chosen here
- **`pfm-v5b-s1-e14` for reference only: not chosen by the rule.** Whatever it shows, the served model does not change
- **SM-2:** within-window AUC ≥ 0.70 on the shifted held-out UCF-Crime, D-65's ranges (`close` τ [8, 13) s, `far`
  τ [20, 30) s). The unshifted 0.5736 stays beside it

1. **Setup:** clone, the pinned versions, Drive
2. **The missing vectors:** the shifted 60 s windows' up to 8 s before the old 60 s window (222 grid points in 31
   videos), OMNI-902's encoder, pins and clip → `embeddings/pfm/ucf_crime/internvideo2_shift8/` (only those
   points; the existing folder is not touched). Resumable
3. **The measures:** both runs on the shifted windows (the two folders read together), the eight measures overall
   and per length; for reading only, the unshifted windows of the same videos → `results/x23_heldout_shift8.json`
4. **SM-2**

Runtime → Change runtime type → **GPU** (a T4 is enough: a few minutes).

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

# The versions in uv.lock (as OMNI-902); Colab keeps its own GPU build of torch and torchvision.
!pip install -q "transformers==5.20.1" "huggingface-hub==2.2.0" "safetensors==0.8.0" "open-clip-torch==3.3.0" -e /content/omni-sight/services/common

sys.path.insert(0, "/content/omni-sight/services/common")
sys.path.insert(0, "/content/omni-sight/services/ai_workers")
sys.path.insert(0, "/content/omni-sight/scripts")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
DEVICE, GPU = "cuda", torch.cuda.get_device_name(0)
print(torch.__version__, GPU)

In [ ]:
# 2 · The missing vectors, OMNI-902's encoder, pins and clip, into a new folder
import json, time
from pathlib import Path
import numpy as np
import build_heldout_shift as shift
import encode_pfm_windows as pfm
import pfm_samples
from omnisight_common.settings import CommonSettings
from app.encoder import ensure_checkpoint
from app import clip_encoder
from app.clip_encoder import ClipTextEncoder

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
UCF = DRIVE / "data" / "ucf_crime"
OLD = DRIVE / "embeddings" / "pfm" / "ucf_crime" / "internvideo2"
NEW = DRIVE / "embeddings" / "pfm" / "ucf_crime" / "internvideo2_shift8"

WINDOWS = pfm.read_windows(REPO_DIR / "datasets" / "windows" / "ucf_crime.csv")
MOVED, NONE = shift.shifted(WINDOWS)
assert shift.to_csv([r for rs in MOVED.values() for r in rs]) == (REPO_DIR / "datasets" / "windows" / "ucf_crime_heldout_shift8.csv").read_bytes()
NEED = shift.missing(WINDOWS, MOVED, pfm.STEPS_S["ucf_crime"])
for key, frames in NEED.items():  # the existing folder really lacks them, and has the rest
    with np.load(pfm.out_file(OLD, key)) as f:
        assert not set(frames) & set(f["frame_index"].tolist()), key
print(f"{len(MOVED)} videos shifted, {len(NONE)} windows without one; missing {sum(map(len, NEED.values()))} points in {len(NEED)} videos")

pins = {name: CommonSettings.model_fields[name].default for name in ("clip_model_id", "clip_model_revision")}
assert pins == json.loads((OLD / pfm.MANIFEST).read_text())["pins"]  # the same encoder as the existing folder
SESSION = time.strftime("%Y-%m-%d %H:%M") + " " + GPU
model_dir = ensure_checkpoint(pins["clip_model_id"], pins["clip_model_revision"], DRIVE / "models" / "pretrained" / "internvideo2-clip-s", clip_encoder.ALLOW_PATTERNS)
encoder = ClipTextEncoder(model_dir, device="cuda")
manifest = pfm.run({k: MOVED[k] for k in NEED}, UCF, NEW, encoder, clip_encoder, pins, pfm.STEPS_S["ucf_crime"],
                   context_s=0.0, session=SESSION, only=NEED)
ENCODING = pfm.summary({k: MOVED[k] for k in NEED}, NEW)
print(ENCODING)
assert ENCODING["done"] == len(NEED) and not ENCODING["failed"]
for key, frames in NEED.items():
    with np.load(pfm.out_file(NEW, key)) as f:
        assert f["frame_index"].tolist() == frames, key

In [ ]:
# 3 · The measures: the served run and B's reference on the shifted windows; the unshifted on the same videos
import hashlib
import pfm_heldout, pfm_runs, train_pfm

LOCAL = Path("/content/pfm")
ALL = pfm_samples.read_samples(REPO_DIR / "datasets" / "samples" / "ucf_crime.csv")
HELD_NORMAL = [s for s in ALL if s["split"] == "held_out" and s["label"] == "Normal"]
SHIFTED = pfm_samples.read_samples(REPO_DIR / "datasets" / "samples" / "ucf_crime_heldout_shift8.csv")
SAME_VIDEOS = [s for s in ALL if s["split"] == "held_out" and s["label"] != "Normal" and s["clip_key"] in MOVED]
INPUTS = {"shifted": HELD_NORMAL + SHIFTED, "unshifted, same videos": HELD_NORMAL + SAME_VIDEOS}
STORE = {"shifted": pfm_samples.Store(INPUTS["shifted"], OLD, extra_dir=NEW),
         "unshifted, same videos": pfm_samples.Store(INPUTS["unshifted, same videos"], OLD)}
RUNS = {"pfm-v5a-s1-e14": "served (D-65's rule)", "pfm-v5b-s1-e14": "reference only, NOT chosen by the rule"}
assert pfm_runs.download(DRIVE, None, target=LOCAL / "chosen")["run_id"] == "pfm-v5a-s1-e14"  # the chosen run is unchanged

KEYS = [("within_window_auc", ".4f"), ("flat_line", ".4f"), ("mtta_s", ".2f"), ("detection_rate", ".3f"),
        ("false_alarm_windows", ".3f"), ("false_alarms_per_hour", ".1f"), ("ap", ".4f"), ("per_window_auc", ".4f"),
        ("pooled_auc", ".4f")]
v = lambda x, spec=".4f": "-" if x is None else format(x, spec)
def line(name, s):
    return (f"  {name:28s} " + "  ".join(f"{k} {v(s[k], spec)}" for k, spec in KEYS)
            + f"  windows {s['within_window_windows']}+{s['within_window_skipped']} skipped / "
              f"{s['flat_line_windows']}+{s['flat_line_skipped']} skipped")

RESULTS = {}
for rid, role in RUNS.items():
    run = pfm_runs.download(DRIVE, rid, target=LOCAL / rid)
    model, state = train_pfm.load(LOCAL / rid / pfm_runs.CHECKPOINT, DEVICE)
    theta = run["thresholds"]["ucf_crime"]
    print(f"\n{rid} ({role}), epoch {state['epoch']}, θ {theta:.4f}")
    RESULTS[rid] = {"role": role, "theta": theta, "checkpoint_sha256": run["checkpoint_sha256"]}
    for name, samples in INPUTS.items():
        data = train_pfm.Data(STORE[name], samples, range(len(samples)), "ucf_crime").for_labels(model.config.labels)
        p, dt = train_pfm.predict(model, data, DEVICE)
        r = pfm_heldout.measure(data, p, dt, theta, 5.0)
        RESULTS[rid][name] = {k: x for k, x in r.items() if k != "windows"}
        print(line(name, r["summary"]))
        for length, s in r.get("per_length", {}).items():
            print(line(f"{name}, {length} s", s))

out = DRIVE / "results" / "x23_heldout_shift8.json"
out.write_text(json.dumps({
    "commit": COMMIT, "gpu": GPU, "torch": torch.__version__, "encoding": ENCODING,
    "windows": {"shifted": sum(map(len, MOVED.values())), "without_one": len(NONE), "videos": len(MOVED)},
    "samples_sha256": {n: hashlib.sha256((REPO_DIR / "datasets" / "samples" / n).read_bytes()).hexdigest()
                       for n in ("ucf_crime.csv", "ucf_crime_heldout_shift8.csv")},
    "results": RESULTS}, indent=1, default=float))
print(out)

In [ ]:
# 4 · SM-2 (D-66): within-window AUC ≥ 0.70 on the shifted held-out UCF-Crime, the served run only
s = RESULTS["pfm-v5a-s1-e14"]["shifted"]["summary"]
u = RESULTS["pfm-v5a-s1-e14"]["unshifted, same videos"]["summary"]
print(f"SM-2: shifted {v(s['within_window_auc'])} (target ≥ 0.70): "
      f"{'pass' if s['within_window_auc'] is not None and s['within_window_auc'] >= 0.70 else 'fail'}; "
      f"unshifted (X22, all videos) 0.5736; unshifted on the same videos {v(u['within_window_auc'])} (for reading only)")
b = RESULTS["pfm-v5b-s1-e14"]["shifted"]["summary"]
print(f"B (reference only, NOT chosen by the rule): shifted {v(b['within_window_auc'])}")